# DocuMind — RAG Document Q&A walkthrough

This notebook demonstrates LlamaIndex document indexing and semantic retrieval, followed by LangChain prompt orchestration and structured response parsing. It uses a small in-memory sample document. Set `OPENAI_API_KEY` before running the answer cell. The first index run downloads a local Hugging Face sentence-embedding model.

Install project dependencies from the repository root with `python -m pip install -r requirements.txt`.

In [ ]:
import os
from llama_index.core import Document
from document_qa import build_index, retrieve_context, answer_question

## Create a multi-document collection

The deployed app can ingest PDFs, UTF-8 TXT/Markdown files, and public HTML/text web pages. This example creates two source documents without requiring local files.

In [ ]:
documents = [
    Document(
        text=("The Northstar Transit pilot began in April 2025 in three districts. "
              "The pilot added 18 electric buses and measured on-time arrivals, energy use, "
              "and rider feedback over six months."),
        metadata={"source": "pilot_report.txt", "page": "1", "kind": "text"},
    ),
    Document(
        text=("The pilot report found that on-time arrivals improved by 8 percentage points "
              "in the participating districts. The report notes that the short study period "
              "and lack of a randomized comparison group limit causal conclusions."),
        metadata={"source": "evaluation_notes.txt", "page": "1", "kind": "text"},
    ),
]
index = build_index(documents)
print(f"Indexed {len(documents)} source documents.")

## Retrieve semantic evidence

LlamaIndex embeds document chunks and returns the nearest passages for the question. The retrieved source IDs are subsequently used as allowed citation identifiers.

In [ ]:
question = "How did punctuality change, and what limits the conclusion?"
passages = retrieve_context(index, question, top_k=4)
for passage in passages:
    print(f"[{passage.source_id}] {passage.source} · page {passage.page} · score={passage.score}")
    print(passage.text, "\n")

## Generate a grounded answer with LangChain

The response chain is `ChatPromptTemplate → ChatOpenAI → PydanticOutputParser`. Retrieved passages are sent to OpenAI for generation. The prompt treats document text as untrusted evidence, requires an insufficient-evidence flag, and validates returned citation IDs against the retrieved passages.

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before generating an answer.")
answer = answer_question(question, passages)
print(answer.answer)
print("Citations:", answer.citations)
print("Insufficient evidence:", answer.insufficient_evidence)

## Try collection summarization

A summarization request is an ordinary query over the same retriever and grounded answer pipeline. The Streamlit UI exposes this as **Summarize sources**.

In [ ]:
summary_question = "Summarize the main topics and key points across these sources."
summary_passages = retrieve_context(index, summary_question, top_k=6)
summary = answer_question(summary_question, summary_passages)
print(summary.answer)
print("Citations:", summary.citations)